# Гіт та інше · Урок 1 — Git та системи контролю версій

👋 Ви вже вмієте писати Python і працювати з БД. Тепер опануємо **інструмент №1 будь-якого розробника** — **Git**. Без нього не обходиться жоден проєкт і жодна співбесіда.

Не хвилюйтесь, якщо на початку Git здається заплутаним — так у всіх. Ми йтимемо **повільно, з аналогіями**, і до кінця уроку картинка складеться.

**Передумова:** Git має бути встановлений — див. [vstanovlennya-git.md](../vstanovlennya-git.md). Базове знання терміналу.

### Що ви вмітимете після уроку
- пояснити **простими словами**, що таке Git і навіщо він;
- впевнено користуватися основними **командами** (і розуміти, що вони роблять «під капотом»);
- **уявляти гілки як діаграму** — і розуміти `merge`, `rebase`, `cherry-pick`;
- рятувати роботу через `stash`, `reflog`, відрізняти `reset`/`revert`/`checkout`;
- пояснити **Gitflow** та інші командні робочі процеси.

> 📌 Позначка **🔎 Важливо знати** — теми, які майже завжди питають на співбесідах.
> Позначка **🚀 Middle+** — глибші теми: Junior-у досить розуміти ідею.

> 🎯 **Питання зі співбесіди**, на які цей урок дає повну відповідь:
> 1. Що таке Git і навіщо він? · 2. Які команди Git ви знаєте? · 3. Що таке `git stash`? ·
> 4. Що таке `git cherry-pick`? · 5. Розкажіть про Gitflow. — кожне позначене 🎯.

## 1. 🔎🎯 Що таке Git і навіщо він потрібен

Уявіть, що ви пишете курсову. Щоб не втратити роботу, ви робите копії теки:
`курсова`, `курсова_фінал`, `курсова_фінал2`, `курсова_фінал_справжній`. Через тиждень ви вже не пам'ятаєте, де що і чим вони відрізняються. А тепер уявіть, що над тим самим проєктом так працюють **п'ять людей**. Хаос.

**Git вирішує саме цю проблему.** Простими словами:

> **Git — це «машина часу» для вашого коду.** Він запам'ятовує кожен збережений стан проєкту, показує, хто/що/коли змінив, і дозволяє будь-коли повернутись назад або об'єднати роботу кількох людей.

Офіційно Git називають **системою контролю версій** (VCS — *version control system*): «версія» = збережений стан проєкту, «контроль» = можливість ним керувати (дивитись, порівнювати, відкочувати).

**Що конкретно це дає (і чому копії теки — гірше):**

- 🕓 **Історія.** Кожен збережений стан (**коміт**) лишається назавжди. Замість десятка тек — **один** проєкт із повним журналом змін. Видно, що і коли змінилось, і можна **відкотитись** до будь-якого моменту.
- 👥 **Спільна робота.** Десятеро людей правлять той самий проєкт паралельно, а Git акуратно **зливає** їхні зміни в одне ціле (та ще й підкаже, якщо двоє правили той самий рядок).
- 🌿 **Гілки.** Нову функцію можна писати в **окремій «паралельній» версії** проєкту, не ламаючи основну робочу. Не вийшло — просто викинули гілку.
- 💾 **Резервність.** Повна копія проєкту (з усією історією!) лежить у **кожного** розробника на комп'ютері **і** на сервері (GitHub). Згорів ноутбук — нічого не втрачено.
- 🔍 **Прозорість.** Команда `git blame` покаже, хто і в якому коміті написав конкретний рядок — не щоб звинувачувати, а щоб знати, у кого спитати.

**Що означає «розподілена» (distributed).** Є два підходи. У старих системах (як SVN) історія жила **лише на сервері** — без інтернету працювати не могли. Git — **розподілений**: у кожного на комп'ютері лежить **повна копія репозиторію з усією історією**. Тому можна спокійно комітити в літаку без Wi-Fi, а коли з'явиться інтернет — синхронізуватись із командою.

```
   ┌────────────┐        ┌──────────── GitHub (сервер у хмарі) ────────┐
   │ Розробник A│  push  │                                              │
   │ (повна     │ ─────► │            origin/main                       │
   │  копія +   │ ◄───── │      (спільна копія з повною історією)       │
   │  історія)  │  pull  │                                              │
   └────────────┘        └──────────────────────────────────────────────┘
                                      ▲        ▲
                                push  │        │  pull
                              ┌───────┘        └────────┐
                        ┌────────────┐            ┌────────────┐
                        │ Розробник B│            │ Розробник C│
                        │(теж повна  │            │(теж повна  │
                        │  копія)    │            │  копія)    │
                        └────────────┘            └────────────┘
```
`push` = «відправити свої коміти на сервер», `pull` = «забрати звідти чужі». Про них — далі.

> ⚠️ **Git ≠ GitHub — це найчастіша плутанина.** Git — це **програма** у вас на комп'ютері (сам інструмент). GitHub (а ще GitLab, Bitbucket) — це **сайт**, який зберігає Git-репозиторії в хмарі й додає зручності: сторінку проєкту, Pull Request'и, ревʼю коду, автоматичні перевірки. Аналогія: **Git — це формат (як «документ»), а GitHub — хмара для нього (як Google Drive)**. Git чудово працює й узагалі без інтернету; GitHub просто дає спільне місце для команди.

## 2. Як Git улаштований: три «зони» та коміт

Це — **найважливіший розділ для розуміння**. Коли ви вловите ідею трьох зон, зникне 90% плутанини з `add` і `commit`.

Коли ви робите `git init` у теці проєкту, Git створює всередині приховану підтеку **`.git`** — це і є весь «мозок» репозиторію (уся історія, налаштування, гілки). Саме її наявність відрізняє звичайну теку від Git-репозиторію.

Далі кожна ваша зміна проходить через **три зони**:

```
 Working Directory        Staging Area (Index)        Repository (.git)
 (ваші файли на диску)     (що ввійде в наступний        (збережена історія
                             коміт — «чернетка»)           комітів)
 ┌───────────────┐  git add  ┌───────────────┐  git commit ┌───────────────┐
 │   main.py  ✎  │ ────────► │   main.py     │ ──────────► │  коміт a1b2c3 │
 │  (змінений)   │           │ (застейджений)│             │  "Add login"  │
 └───────────────┘           └───────────────┘             └───────────────┘
        ▲                                                          │
        └──────────────  git checkout / restore  ──────────────────┘
                         (повернути файл зі сховища)
```

Розберемо кожну зону **по-людськи**:

1. **Working Directory (робоча тека)** — це ваші звичайні файли, які ви прямо зараз редагуєте у VS Code. Усе, що ви змінили, спершу живе тільки тут.

2. **Staging Area (зона підготовки, вона ж index)** — це **«чернетка наступного коміту»**. Аналогія: ви **пакуєте коробку перед відправленням** і самі кладете в неї саме ті речі, які хочете відправити. Командою `git add file.py` ви кладете зміни файлу в цю коробку. Навіщо окремий крок? Щоб можна було зробити **один охайний коміт лише з потрібних змін**, а не звалити все підряд. Наприклад, ви поправили баг *і* дорелагодили щось інше — можна застейджити й закомітити їх **окремо**, двома різними комітами.

3. **Repository (сховище, `.git`)** — коли коробка зібрана, команда `git commit` **запечатує** її: staging перетворюється на **коміт** — незмінний знімок, що назавжди лягає в історію.

**Життєвий цикл файлу** (як Git «бачить» файл):
```
 (новий файл)      (git add)       (git commit)      (знову правите)
 untracked  ──►   staged     ──►   committed    ──►   modified  ──► (git add) ...
 «Git його       «покладено       «збережено в      «є зміни, ще
  ще не знає»      в коробку»       історію»          не в коробці»
```
- **untracked** — файл новий, Git його ще не відстежує;
- **modified** — відстежуваний файл змінено, але зміни ще не в staging;
- **staged** — зміни покладено в коробку (`git add`), готові до коміту;
- **committed** — зміни збережено в історію.

**А що таке сам коміт?** Це **знімок (snapshot) усього проєкту** в конкретний момент + трохи метаданих:
```
коміт  a1b2c3d                     ← унікальний "відбиток" (хеш), як номер коміту
├─ author:  Ваше Імʼя <mail>       ← хто зробив
├─ date:    2026-07-09 14:30       ← коли
├─ message: "Add login form"       ← що зробив (ваш опис)
├─ parent:  9f8e7d6                ← посилання на ПОПЕРЕДНІЙ коміт
└─ знімок усіх файлів на цей момент
```
Два важливі моменти, які варто запам'ятати:
- **Хеш** (`a1b2c3d...`) — це унікальний «відбиток пальця» коміту. Git обчислює його з вмісту; двох однакових не буває. За ним ви посилаєтесь на коміт у командах.
- Кожен коміт **пам'ятає свого батька** (parent). Тому коміти шикуються в **ланцюжок** — і цей ланцюжок і є вашою історією:
```
A ◄── B ◄── C ◄── D        (стрілки — «хто чий батько»; D — найновіший)
```

## 3. 🔎🎯 Які команди Git ви знаєте

Це питання лякає новачків, бо здається, що треба знати сотні команд. Насправді **на 95% роботи вистачає ~15 команд**. На співбесіді не тараторте весь список — назвіть **основні за групами** й покажіть, що розумієте, **коли** яку застосувати. Нижче — ті самі групи з поясненням кожної.

**⚙️ Налаштування (робиться один раз на комп'ютері).** Git має знати, ким підписувати ваші коміти:
```bash
git config --global user.name "Ваше Імʼя"
git config --global user.email "you@example.com"
```

**📦 Створити / отримати репозиторій.** Або починаєте з нуля, або завантажуєте існуючий:
```bash
git init                 # перетворити поточну теку на Git-репозиторій (з'явиться .git)
git clone <url>          # завантажити (склонувати) чужий репозиторій з GitHub на свій комп'ютер
```

**📸 Щоденний цикл — зберегти зміни (це ви робитимете найчастіше):**
```bash
git status               # головна команда новачка: що змінилося, що вже в "коробці"
git add file.py          # покласти зміни файлу в staging (git add .  — покласти геть усе)
git commit -m "текст"    # запечатати staging як коміт із описом
git commit -am "текст"   # скорочення: add усіх ВІДСТЕЖУВАНИХ файлів + commit одним махом
```
> 💡 Звикайте раз у раз писати `git status` — вона завжди підкаже, у якому ви стані й що робити далі.

**🔍 Подивитися історію / порівняти:**
```bash
git log --oneline --graph --all   # історія у вигляді компактного дерева (дуже наочно)
git diff                 # що саме змінилось, але ще не застейджено (порядково)
git show <hash>          # показати конкретний коміт: що в ньому змінилось
git blame file.py        # хто написав кожен рядок файлу (і в якому коміті)
```

**🌿 Гілки (паралельні версії проєкту — детально в розділі 4):**
```bash
git branch               # список гілок (зірочка * — де ви зараз)
git branch feature-x     # створити гілку (але лишитись на поточній)
git switch feature-x     # перейти на гілку feature-x (сучасний варіант)
git checkout -b feature-x   # те саме, старіший варіант
git switch -c feature-x  # створити гілку І одразу перейти на неї (сучасний варіант)
git checkout -b feature-x # те саме, старіший варіант (create + switch)
git merge feature-x      # влити зміни гілки feature-x у поточну гілку
git branch -d feature-x  # видалити гілку, яка вже не потрібна
```

> 💡 **`switch`/`restore` проти `checkout`.** Раніше все робив один перевантажений `git checkout`: і перемикав гілки, і відновлював файли — новачки плутались. У Git 2.23+ його «розділили надвоє»: `git switch` — для гілок, `git restore` — для файлів. Тому це рівнозначно: `git switch -c нова` = **`git checkout -b нова`** (створити гілку **і одразу перейти**). `checkout -b` досі всюди зустрічається у статтях і в колег, тож знати його обов'язково.

**🌐 Робота з сервером (remote — детально в розділі 9):**
```bash
git remote -v            # які віддалені репозиторії підключено (зазвичай origin = GitHub)
git fetch                # ЗАВАНТАЖИТИ чужі зміни з сервера, але поки НЕ зливати у свою гілку
git pull                 # fetch + merge: завантажити І одразу влити у свою гілку
git push                 # відправити СВОЇ коміти на сервер
git push -u origin main  # перший push гілки: відправити + запам'ятати зв'язок (далі просто git push)
```

**↩️ Скасувати / виправити (коли щось пішло не так — розділ 8):**
```bash
git restore file.py      # відкинути незакомічені зміни у файлі (повернути як було)
git reset <hash>         # зсунути гілку назад в історії (є 3 режими — нижче)
git revert <hash>        # створити НОВИЙ коміт, що скасовує зміни іншого (безпечно)
git commit --amend       # виправити ОСТАННІЙ коміт (напр. одрук у повідомленні)
```

> 💬 **Як відповісти на співбесіді (формула):** «Щодня — `status`, `add`, `commit`, `push`, `pull`. Гілки — `branch`, `switch`, `merge`. Дивитись історію — `log`, `diff`, `blame`. Скасувати — `restore`, `reset`, `revert`. А з глибших знаю `stash`, `rebase`, `cherry-pick`, `reflog`.» Далі ми розберемо кожну з них 👇

## 4. Гілки як діаграма — головна інтуїція

Слово «гілка» лякає, а ідея — проста. **Гілка — це паралельна версія проєкту**, у якій можна щось пробувати, не чіпаючи основну (`main`). Аналогія з фільмів: це **паралельний всесвіт** вашого коду. Пішло добре — «зливаєте» його назад в основну реальність. Пішло погано — просто викидаєте, основна лишилась цілою.

Тепер найважливіше — **що таке гілка «під капотом»**. Багато хто думає, що це «копія всіх файлів». Це не так, і саме тому гілки в Git такі швидкі:

> **Гілка — це просто рухома мітка (вказівник), яка показує на один коміт.** Технічно це крихітний файлик у `.git` із хешем коміту всередині. Ось чому створити гілку — миттєво: Git не копіює файли, він лише ставить нову мітку.

Ще один важливий термін — **`HEAD`**. Це «**ви тут**» — мітка, що показує, на якій гілці/коміті ви зараз перебуваєте. Коли ви робите новий коміт, рухається саме та гілка, на яку показує `HEAD`.

Уявляймо історію як **граф комітів**, а гілки — як мітки на ньому.

**Крок 1. Лінійна історія (одна гілка `main`):**
```
                  HEAD
                   │
                   ▼
  A ──► B ──► C ──► D        ◄── main
 (час зліва направо; кожен коміт пам'ятає свого батька)
```
`main` показує на `D` (останній коміт), а `HEAD` показує на `main`. Усе просто.

**Крок 2. Створюємо гілку `feature` — з'являється нова мітка на тому ж коміті:**
```bash
git switch -c feature    # створити гілку feature й перейти на неї
```
```
                  HEAD
                   │
                   ▼
                feature
                   │
  A ── B ── C ── D ◄── main
 (обидві мітки, feature і main, поки що на одному коміті D)
```
Зверніть увагу: файли **не копіювались**. Просто з'явилась ще одна мітка `feature`, а `HEAD` переїхав на неї (тепер «ви тут» = на feature).

**Крок 3. Робимо 2 коміти в `feature` — гілки розходяться:**
```
                       HEAD
                        │
                        ▼
                E ── F ◄── feature
               /
  A ── B ── C ── D      ◄── main
```
Дивіться, що сталось:
- `feature` «поповзла» вперед у коміти `E`, потім `F` (бо `HEAD` був на ній — рухалась саме вона);
- `main` **лишилась на `D`** — ми ж на ній не комітили;
- вийшли **дві розбіжні лінії**. Основна версія (`main`) недоторкана, а вся ваша нова робота — у `feature`.

Саме тому в Git **не бояться гілок** і створюють нову на кожну задачу: це дешево (лише мітка) і безпечно (основна версія не страждає).

> 🚀 **Middle+: detached HEAD («відірваний HEAD»).** Якщо зробити `git checkout <хеш коміту>` (не гілки, а прямо коміту), `HEAD` покаже **напряму на коміт**, а не на гілку. Ви можете дивитись/пробувати, але нові коміти тут «нічиї» — їх легко втратити. Git попередить. Щоб зберегти роботу — створіть гілку: `git switch -c нова-гілка`.

> 🧠 **Запам'ятати три речі:** коміт **незмінний** і знає свого **батька**; гілка — це **мітка, що їде вперед** при новому коміті; `HEAD` — це «**ви тут**».

## 5. Об'єднання гілок: `merge` проти `rebase`

Ви написали фічу в гілці `feature`. Тепер її треба **повернути** в `main`, щоб вона стала частиною основного проєкту. Це називається **об'єднання (integration)**, і є два способи: `merge` і `rebase`. Розберемо обидва повільно.

### Спосіб 1: `merge` («злиття»)

**Випадок А — fast-forward («перемотування вперед»).** Найпростіший. Якщо поки ви робили фічу, в `main` **ніхто нічого не додав** (вона так і стоїть на `D`), то Git не має що «зливати» — він просто **пересуває мітку `main` вперед**, на ваш останній коміт. Ніби перемотує касету вперед.
```
до:                              після  git switch main; git merge feature:
        E ── F ◄─ feature                       E ── F ◄─ feature, main
        /                                        /
A ─ B ─ C ◄─ main                     A ─ B ─ C
                                      (мітка main просто "доїхала" до F)
```

**Випадок Б — 3-way merge (справжнє злиття).** А якщо поки ви робили фічу, **хтось інший теж комітив у `main`** (вона поповзла в `D`)? Тоді обидві лінії пішли вперед, і Git не може просто «перемотати». Він бере два кінці (`D` і `F`), їхнього спільного предка (`C`) і **сплітає їх в один новий коміт злиття `M`** — особливий коміт, у якого **два батьки**.
```
        E ── F                         E ─── F
       /       \                      /       \
A ─ B ─ C ─── D   ◄─ main    A ─ B ─ C ─── D ── M   ◄─ main
                                          (M — merge-коміт, батьки: D і F)
```
```bash
git switch main
git merge feature        # створить merge-коміт M
```
- ➕ Історія **чесна** — видно, що фіча розроблялась паралельно й коли влилась.
- ➖ Якщо таких злиттів багато, історія стає «кучерявою», з купою merge-комітів.

### Спосіб 2: `rebase` («перенесення основи»)

`rebase` вирішує ту саму задачу інакше. Замість зливати, він ніби каже: «а давай уявимо, що я почав робити фічу **не від старого `C`, а від найсвіжішого `D`**». Git бере ваші коміти `E`, `F`, **знімає їх** і **накладає заново** поверх `D`. Виходить рівна пряма лінія — ніби ви весь час працювали від найновішої версії.
```
до:                                 після  git switch feature; git rebase main:
      E ── F ◄─ feature                            E'── F' ◄─ feature
     /                                            /
A ─ B ─ C ─ D ◄─ main               A ─ B ─ C ─ D ◄─ main
   (E,F виросли з C)                   (E,F перенесені на D як НОВІ коміти E', F')
```
Зверніть увагу на **штрихи**: `E'` і `F'` — це **нові** коміти (з новими хешами), навіть якщо код у них той самий. Старі `E`, `F` Git «переписав».
- ➕ Історія **лінійна й чиста** — читається як послідовна розповідь, без розгалужень.
- ➖ Переписування хешів. І звідси — головне застереження:

> ⚠️ **Золоте правило `rebase`.** Ніколи не робіть `rebase` гілки, яку вже **запушили й нею користуються інші**. Ви переписуєте коміти (нові хеші) — а в колег лишились старі, і їхня історія «розсинхронізується» з вашою. Наслідок — біль і конфлікти. Просте правило: **rebase — лише для своїх локальних, ще не опублікованих комітів.** Спільне об'єднуйте через `merge`.

| | `merge` | `rebase` |
|---|---|---|
| Що робить | сплітає гілки merge-комітом | переносить коміти на новішу основу |
| Історія | розгалужена, «як було» | лінійна, чиста |
| Створює нові хеші | ні | так (переписує) |
| Безпечно для спільних гілок | ✅ так | ❌ ні |

> 💬 **Коротка відповідь:** «`merge` зливає гілки й лишає merge-коміт із двома батьками — історія розгалужена, зате чесна. `rebase` переносить мої коміти поверх свіжої гілки й робить лінійну історію, але переписує хеші, тож його не можна застосовувати до вже опублікованих спільних гілок.»

## 6. 🔎🎯 `git stash` — тимчасово відкласти зміни

**Життєва ситуація.** Ви посеред роботи над фічею — купа незакомічених, ще «сирих» змін. І тут прибігає керівник: «терміново треба полагодити баг на проді, прямо зараз!». Проблема: щоб перемкнутись на іншу гілку, робоча тека має бути чистою, а комітити напівроблений код не хочеться (він же не готовий).

**Рішення — `git stash`.** Аналогія: ви **згрібаєте всі незавершені зміни в шухляду**, стіл стає чистим, ви робите термінову справу, а потім **дістаєте зміни назад** і продовжуєте, ніби нічого не було.

```
робоча копія (брудна)                       робоча копія (чиста)
 ┌──────────────┐   git stash    ┌────────┐   ┌──────────────┐
 │ незакомічені │ ─────────────► │ ШУХЛЯДА│   │  все чисто →  │
 │    зміни     │                │(сховок)│   │  можна        │
 └──────────────┘                └────────┘   │  перемикатись │
        ▲                            │        └──────────────┘
        └────────── git stash pop ───┘  (повернути зміни назад)
```

**Основні команди:**
```bash
git stash                 # згребти зміни в шухляду (робоча копія стає чистою)
git stash list            # що лежить у шухляді (stash@{0} — найсвіжіший, stash@{1} старіший...)
git stash pop             # ДІСТАТИ останній сховок назад і прибрати його з шухляди
git stash apply           # дістати, але ЛИШИТИ копію в шухляді (про запас)
git stash drop            # викинути сховок із шухляди
git stash -u              # згребти ще й нові (untracked) файли, які Git ще не відстежує
git stash push -m "опис"  # сховок із підписом, щоб потім упізнати
```

Кілька деталей, які варто знати:
- Шухляда — це **стек**: можна відкладати кілька разів; сховки складаються один на одного, а `pop` дістає **найсвіжіший**.
- `pop` = «дістати **і** прибрати зі стеку», `apply` = «дістати, але лишити копію». Якщо не впевнені — `apply` безпечніше.
- Типовий сценарій цілком: `git stash` → перемкнутись на іншу гілку → полагодити баг, закомітити → повернутись на свою гілку → `git stash pop` → спокійно продовжити фічу.

> 💬 **Коротка відповідь на співбесіді:** «`git stash` тимчасово ховає незакомічені зміни в окремий стек і очищає робочу копію, щоб можна було перемкнути контекст (напр. терміново полагодити щось в іншій гілці). Повертаю зміни через `git stash pop`.»

## 7. 🔎🚀🎯 `git cherry-pick` — взяти лише один коміт

Назва каже сама за себе: **cherry-pick** — це «зірвати одну вишеньку». Уся гілка вам не потрібна — треба лише **один конкретний коміт** з неї перенести в поточну гілку. Саме це і робить `git cherry-pick <хеш>`: бере зміни з одного коміту й **застосовує їх** там, де ви зараз.

```
feature:   A ─ B ─ C ─ D ─ E          нам треба лише коміт D, а не вся гілка
                       │
                       ▼  git switch main; git cherry-pick <хеш D>
main:      A ─ B ─ X ─ D'             ◄─ D' — КОПІЯ коміту D (з новим хешем!)
```
(Так само, як `rebase`, cherry-pick робить **копію** коміту — новий хеш `D'`, бо він тепер стоїть на іншому місці історії.)

**Коли це реально стається на роботі:**
- терміновий фікс зробили у фіча-гілці, а він потрібен **уже зараз у `main`** — не чекати ж, поки допишеться вся фіча;
- ви випадково закомітили не в ту гілку — переносите потрібний коміт куди слід;
- у релізну гілку треба «підібрати» лише кілька конкретних виправлень, а не все підряд.

```bash
git switch main
git cherry-pick a1b2c3d           # застосувати один коміт
git cherry-pick a1b2c3d f4e5d6c   # застосувати кілька (по черзі)
git cherry-pick A..B              # цілий діапазон комітів
```

> ⚠️ **Не зловживайте.** Оскільки cherry-pick створює **копію** з іншим хешем, той самий зміст може згодом потрапити в історію **двічі** (коли ви пізніше зіллєте гілки повністю) — це плутанина й потенційні конфлікти. Тому cherry-pick — це **точковий інструмент** («ось саме цей фікс — сюди»), а не заміна нормального `merge`.

> 💬 **Коротка відповідь:** «`cherry-pick` застосовує **один** конкретний коміт з іншої гілки в поточну — зручно, коли треба лише окремий фікс, а не вся гілка. Він робить копію коміту з новим хешем, тож зловживати ним замість merge не варто.»

## 8. 🔎🎯 Gitflow та інші командні робочі процеси (workflows)

Команди Git ми вивчили. Але в команді постає інше питання: **скільки має бути гілок і хто як через них працює?** Домовленість про це називають **workflow** («робочий процес»). Це не команда Git, а **правила гри**, які команда обирає для себе.

### Gitflow — класична «сувора» модель із кількома постійними гілками

Уявіть **конвеєр на заводі**: сира деталь проходить кілька цехів, перш ніж стати готовим товаром. Gitflow влаштований схоже — у нього дві **постійні** гілки й три типи **тимчасових**.

```
                         тег v1.0            тег v1.1
main      ●───────────────●──────────────────●───────────►  (лише готові релізи, завжди стабільна)
           \             / \                / ▲
hotfix      \           /   \      ●──●─────/  │ (термінові фікси прод-багів)
             \         /     \    /            │
release       \       /       ●──●  release/1.1 (підготовка релізу: фінальні тести, багфікси)
               \     /        /
develop  ●──────●───●────●───●──────●──────────●───────────►  (сюди зливають усі готові фічі)
          \        /    \        \ /
feature    ●──●───●      ●──●──●   ●  feature/*  (кожна нова фіча — окрема гілка)
```

Розберемо ролі гілок **по черзі**:
- **`main`** — «вітрина магазину»: тут лежать **тільки готові, протестовані релізи**. Вона завжди стабільна, з неї деплоять на прод. Кожен реліз позначають **тегом** (`v1.0`, `v1.1`) — про теги далі.
- **`develop`** — «складальний цех»: сюди зливають **усі готові фічі**. Це «наступна версія в роботі». Вона може бути ще нестабільною.
- **`feature/*`** — під **кожну задачу** заводять окрему гілку від `develop`, роблять фічу й вливають назад у `develop`. Приклад: `feature/login`, `feature/cart`.
- **`release/*`** — коли назбирали фіч на реліз, відгалужують `release/1.1`: тут **лише фінальне шліфування** (тести, дрібні багфікси), нових фіч уже не додають. Готово — вливають у `main` (реліз!) **і** назад у `develop`.
- **`hotfix/*`** — на проді горить баг? Відгалужуються **прямо від `main`**, швидко лагодять і вливають назад **і в `main`, і в `develop`** (щоб фікс не загубився в наступному релізі).

➕ Чіткий порядок, добре для великих команд і релізів «за розкладом». ➖ **Складно** — багато гілок і правил; для маленького проєкту чи безперервних деплоїв це надлишок.

### Простіші альтернативи (частіше в сучасних командах)

- **GitHub Flow** — мінімалізм: є лише `main` + короткоживучі `feature`-гілки. Зробив фічу → відкрив **Pull Request** → колеги зробили ревʼю → влили в `main` → задеплоїли. Просто й швидко. **Саме так працюємо ми в цьому курсі** (див. [CONTRIBUTING.md](../CONTRIBUTING.md)).
- **Trunk-Based Development** — усі комітять у `main` (його звуть «trunk», стовбур) маленькими шматочками дуже часто, а незакінчене ховають за «перемикачами» (feature flags). Це основа справжнього CI/CD у великих компаніях.

> 💬 **Коротка відповідь:** «Gitflow — модель із постійними гілками `main` і `develop` та тимчасовими `feature/*`, `release/*`, `hotfix/*`; `main` завжди стабільна, фічі йдуть через `develop`. Добре для релізів за розкладом. Але для більшості проєктів сьогодні беруть простіший **GitHub Flow**: `main` + feature-гілки + Pull Request.»

## 9. 🔎 Інші питання й команди, які люблять питати

Тут — короткі, але часті теми зі співбесід. Розжовуємо кожну.

### `reset` vs `revert` vs `restore` — три способи «скасувати», які плутають
Головна різниця — **чи переписуємо історію** і **що саме відкочуємо**:

```
git revert <hash>   →  створює НОВИЙ коміт, який скасовує зміни зазначеного.
                       Історія ЗБЕРІГАЄТЬСЯ (нічого не зникає) → безпечно у спільних гілках.
                       Було:  A─B─C      Стало:  A─B─C─C'   (C' відміняє C)

git reset  <hash>   →  ЗСУВАЄ мітку гілки назад в історію (ніби «нічого й не було»).
                       Переписує історію. Має 3 режими — чим "глибше", тим більше стирає:
     --soft   зсунути гілку, а зміни лишити В STAGING (готові до нового коміту)
     --mixed  (за замовч.) зсунути, зміни лишити в робочій копії, але прибрати зі staging
     --hard   зсунути й ЗНИЩИТИ всі зміни  ⚠️ незакомічене зникне БЕЗ вороття

git restore <file>  →  просто відкинути зміни в ОДНОМУ файлі (повернути як в останньому коміті).
```
> **Просте правило:** у **спільній** гілці скасовуйте через `revert` (нічого не переписує). `reset --hard` — лише локально й тільки коли точно розумієте, що робите (він реально видаляє роботу).

### `fetch` vs `pull` — у чому різниця
```
git fetch  →  тільки ЗАВАНТАЖИТИ свіжі дані з сервера (оновиться origin/main),
              але вашу робочу гілку НЕ чіпати. Можна спокійно подивитись, що там нового.
git pull   →  це git fetch + git merge разом: завантажити І одразу влити у вашу гілку.
```
Тобто `pull` — «зручно, але одразу міняє мою гілку», а `fetch` — «спершу гляну, потім вирішу». Досвідчені часто роблять `fetch`, дивляться, і лише тоді зливають.

### Конфлікти злиття (merge conflicts) — не страшно
Конфлікт виникає, коли дві гілки змінили **той самий рядок** того самого файлу — Git не знає, чий варіант правильний, і чесно питає вас. Він вставляє в файл **маркери**:
```
<<<<<<< HEAD
твій варіант (те, що в поточній гілці)
=======
варіант з іншої гілки
>>>>>>> feature
```
Що робити: відкрити файл, **лишити правильний код**, прибрати всі три рядки-маркери (`<<<<<<<`, `=======`, `>>>>>>>`), потім `git add file` і `git commit`. Усе — конфлікт розв'язано. Часто редактор (VS Code) ще й підсвітить це кнопками «Accept Current / Accept Incoming».

### Ще команди «на поговорити»
```bash
git tag v1.0                    # позначити реліз тегом (закладка на важливому коміті)
git rebase -i HEAD~3            # інтерактивно причесати останні 3 коміти (об'єднати/перейменувати)
git reflog                      # журнал УСІХ переміщень HEAD — рятує "втрачені" коміти
git bisect                      # бінарний пошук коміту, що вніс баг (дуже потужно)
git clean -fd                   # видалити невідстежувані файли/теки (обережно!)
.gitignore                      # файл зі списком того, що Git має ІГНОРУВАТИ (напр. .venv, __pycache__)
```

> 🚀 **Middle+ — про `reflog`, який рятує життя.** Здається, що після `git reset --hard` чи «загубленого» коміту робота зникла назавжди. Майже ніколи! Git ~30 днів пам'ятає всі рухи `HEAD` у `git reflog`. Знаходите там потрібний хеш — і повертаєте роботу (`git reset --hard <хеш>` або `git switch -c рятунок <хеш>`). Тож перше правило при паніці: **спершу `git reflog`, а вже потім переживати.**

## 10. Типовий робочий цикл (як це виглядає щодня)

Складімо все докупи. Ось як виглядає звичайний день роботи над задачею — крок за кроком:

```bash
git switch main && git pull          # 1. стати на main і забрати свіже з сервера
git switch -c nyaiko/feature-login   # 2. створити гілку під СВОЮ задачу й перейти на неї
#   ... пишемо код у редакторі ...
git status                           # 3. глянути, що змінилось
git add .                            # 4. скласти зміни в "коробку"
git commit -m "Add login form"       # 5. запечатати коміт з описом
git push -u origin nyaiko/feature-login   # 6. відправити гілку на GitHub
#   7. на GitHub натиснути "Compare & pull request" → відкрити Pull Request
#   8. колеги роблять ревʼю → ви правите за зауваженнями → гілку вливають у main
```
Кроки 3–5 (status → add → commit) ви повторюватимете багато разів на день — це і є «щоденний цикл» з розділу 3.

> 📛 **Гарні повідомлення комітів** — це важливо, їх читатимуть інші (і ви ж). Правила прості: **коротко, у наказовому способі, по суті**:
> - ✅ добре: `Add user login`, `Fix null pointer in cart`, `Refactor DB layer`
> - ❌ погано: `fix`, `changes`, `asdf`, `final final2`, `.`
>
> Уявляйте, що речення продовжує фразу «Цей коміт має…»: «*Цей коміт має…* **Add user login**» — звучить логічно.

Саме за цим процесом працюють студенти цього курсу — правила назв гілок і Pull Request'ів описані в [CONTRIBUTING.md](../CONTRIBUTING.md).

## 11. 🔧 Що робити, якщо `git push` не працює (токени й доступ)

Дуже часта ситуація: локально все чудово, а `git push` раптом лається на пароль або доступ. Розберемо повільно, **чому** так і **як** полагодити.

### Чому push просить пароль — і не приймає його
Коли ви пушите на GitHub через **HTTPS** (адреса виду `https://github.com/...`), Git просить логін і пароль. Але GitHub **ще у 2021 році вимкнув вхід за звичайним паролем** — це небезпечно. Замість пароля тепер треба **персональний токен доступу (Personal Access Token, PAT)** — довгий «ключ-пароль», який ви створюєте в налаштуваннях GitHub. Типова помилка виглядає так:
```
remote: Support for password authentication was removed on August 13, 2021.
fatal: Authentication failed for 'https://github.com/...'
```
Простими словами вона каже: «паролем не можна — дай токен».

### ✅ Спосіб 1 (найпростіший) — увійти через gh CLI
Якщо встановлений GitHub CLI (`gh`), він усе зробить за вас — нічого копіювати вручну:
```bash
gh auth login       # обрати: GitHub.com → HTTPS → Login with a web browser
gh auth status      # перевірити, що ви залогінені
```
Після цього `git push` просто працюватиме — `gh` збереже доступ у системі. **Для навчання рекомендую саме цей шлях.**

### Спосіб 2 — створити токен (PAT) вручну
1. GitHub → **Settings** → **Developer settings** → **Personal access tokens** → **Tokens (classic)** → **Generate new token**.
2. Дайте назву, оберіть термін дії й **обов'язково поставте галочку `repo`** (це право читати/писати ваші репозиторії).
3. **Generate** → **скопіюйте токен одразу**: GitHub покаже його лише **раз**, потім не підгляне ніхто.
4. Коли `git push` попросить пароль — **вставте токен замість пароля** (а логін — ваш нікнейм).

### Щоб не вводити токен щоразу — збережіть його
Git уміє запам'ятати доступ у системному сховищі («credential helper»), щоб не питати щоразу:
```bash
# macOS — зберегти в Keychain:
git config --global credential.helper osxkeychain
# Windows — зазвичай уже налаштовано (Git Credential Manager)
# Linux — тримати в пам'яті 15 хвилин:
git config --global credential.helper cache
```
Тоді токен введете **один раз**, а далі Git підставлятиме його сам.

### Спосіб 3 — SSH-ключі (взагалі без токенів)
Замість HTTPS можна ходити на GitHub через **SSH** — і тоді жодних паролів чи токенів при кожному пуші:
```bash
ssh-keygen -t ed25519 -C "you@example.com"   # створити ключ (тисніть Enter на всі питання)
cat ~/.ssh/id_ed25519.pub                     # показати ПУБЛІЧНИЙ ключ і скопіювати його
# GitHub → Settings → SSH and GPG keys → New SSH key → вставити скопійоване
git remote set-url origin git@github.com:USER/REPO.git   # переключити remote на SSH
```

### 🩺 Часті помилки й що робити
| Помилка / симптом | Причина й рішення |
|---|---|
| `Support for password authentication was removed` | Вводите пароль замість токена → створіть PAT (Спосіб 2) або зробіть `gh auth login`. |
| `Authentication failed` | Токен неправильний або **прострочений** → згенеруйте новий і оновіть збережений. |
| `403 Forbidden` / `Permission denied` | Немає прав запису в цей репозиторій **або** токен без галочки `repo`. Це чужий репозиторій? Зробіть **fork** і пуште у свою копію, або попросіть у власника доступ. |
| Git тягне **старий/чужий** акаунт | У сховищі застряг старий токен → приберіть його (macOS: застосунок «Keychain Access» → знайти `github.com`; Windows: «Диспетчер облікових даних» → GitHub). |
| `could not read Username ... terminal prompts disabled` | Немає збереженого доступу → налаштуйте credential helper або перейдіть на SSH. |

> 💡 **Підсумок:** для навчання найлегше — **`gh auth login`** (один раз і забули). Багато хто згодом переходить на **SSH-ключі**. А **токен (PAT)** — універсальний запасний варіант, коли `gh` недоступний. І пам'ятайте: **токен — це пароль**, не викладайте його в код і не показуйте нікому.

# ✅ Підсумок уроку

- **Git — «машина часу» для коду:** розподілена система контролю версій. Дає історію, спільну роботу, гілки, відкат. **Git ≠ GitHub** (інструмент на комп'ютері проти хмарного хостингу).
- **Три зони:** Working Directory → (`git add`) → Staging («коробка») → (`git commit`) → Repository. Коміт — незмінний знімок + вказівник на батька + хеш.
- **Життєвий цикл файлу:** untracked → staged → committed → modified → …
- **Команди щодня:** `status`, `add`, `commit`, `push`, `pull`; гілки — `branch`/`switch`/`merge`; історія — `log`/`diff`/`blame`.
- **Гілка** — це лише **рухома мітка** на коміт (тому дешева); `HEAD` — «ви тут». Уявляйте історію як **граф**.
- **`merge`** зливає гілки (merge-коміт, чесна історія) — безпечно для спільних; **`rebase`** переносить коміти на нову основу (лінійна історія, але переписує хеші — не для опублікованих гілок).
- **`git stash`** — згребти незакомічені зміни в «шухляду»; повернути `stash pop`.
- **`git cherry-pick`** — взяти **один** коміт з іншої гілки (копія з новим хешем).
- **Gitflow** — `main`/`develop`/`feature`/`release`/`hotfix`; простіше й частіше — **GitHub Flow** (main + feature + PR).
- **Скасування:** `revert` (безпечно, новий коміт) vs `reset` (переписує, є soft/mixed/hard) vs `restore` (один файл); `reflog` рятує втрачене.
- **Автентифікація:** `git push` на GitHub через HTTPS вимагає **токен (PAT)**, а не пароль; найпростіше — `gh auth login`. Помилки `403`/`Authentication failed` = немає прав або прострочений токен.

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-1.ipynb](domashnie-zavdannia-lektsiya-1.ipynb)

### ▶️ Далі
Урок 2 — **якість коду**: PEP 8, type hints, docstrings, принципи DRY/KISS/YAGNI та інструменти
(flake8, black, isort, mypy, ruff) і pre-commit hooks.

### 📚 Хочу знати більше
- **Інтерактивна візуалізація гілок** (must-see, укр.): <https://learngitbranching.js.org/?locale=uk>
- Книга Pro Git (безкоштовна, укр.): <https://git-scm.com/book/uk/v2>
- Довідник команд Git: <https://git-scm.com/docs>
- Порівняння workflow (Atlassian): <https://www.atlassian.com/git/tutorials/comparing-workflows>
- Токени доступу GitHub (PAT): <https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens>
- Як писати commit-повідомлення: <https://www.conventionalcommits.org/>